In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, average_precision_score, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


## 1) Loading + basic cleaning

In [ ]:
df = pd.read_csv("data_clean.csv", parse_dates=["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

# Fix your column spellings (your cleaned file removed degree symbol)
# (Optional) Rename to ML-friendly names
rename_map = {
    "Temperature, C": "Temp_C",
    "Humidity, %": "RH_pct",
    "Illumination, lx": "Lux",
    "CO, ppm": "CO_ppm",              # If this is actually CO2, rename to CO2_ppm
    "Pressure, MPa": "Pressure_MPa",
    "Solution temperature, C": "SolTemp_C",
    "Solution acidity, pH": "pH",
    "Solution conductivity, S/cm": "Cond_Scm",
    "Battery, V": "Battery_V",
    "Air circulation": "Air_circulation",
    "Vcc, V": "Vcc_V",
}
df = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns})

df.head()


,Timestamp,Temp_C,RH_pct,Lux,CO_ppm,Pressure_MPa,SolTemp_C,pH,Cond_Scm,Heater,...,Air_circulation,Ground,Fog,Pump 1,Pump 2,Valve 1,Valve 2,Battery_V,Vcc_V,Version
0,2025-05-17 12:49:00,26.7,74,5855,1394.0,0.315,13.8,7.0,406,0,...,0,0,0,1,0,0,0,4.068,11.946,2.6/1.8
1,2025-05-17 12:50:00,27.4,74,6828,1402.0,0.314,14.2,7.0,408,0,...,0,0,0,0,0,0,0,4.068,11.946,2.6/1.8
2,2025-05-17 12:51:00,28.1,73,6802,1397.0,0.010,14.2,7.0,408,0,...,0,0,1,0,0,0,0,4.068,11.964,2.6/1.8
3,2025-05-17 12:52:00,28.4,72,6760,1387.0,0.010,14.3,7.0,404,0,...,0,0,1,0,0,0,0,4.068,11.946,2.6/1.8
4,2025-05-17 12:53:00,29.0,71,9205,1397.0,0.010,14.1,7.0,409,0,...,0,0,1,1,0,0,0,4.068,11.942,2.6/1.8


## 2) Feature set + targets

In [ ]:
target_cols = ["Lighting", "Ventilation", "Air_circulation", "Heater", "Fog"]

# Sensor inputs (exclude Timestamp, targets, and Version)
drop_cols = ["Timestamp", "Version"] + target_cols
X_cols = [c for c in df.columns if c not in drop_cols]

print("X_cols:", X_cols)
print("Targets:", target_cols)


X_cols: ['Temp_C', 'RH_pct', 'Lux', 'CO_ppm', 'Pressure_MPa', 'SolTemp_C', 'pH', 'Cond_Scm', 'Ground', 'Pump 1', 'Pump 2', 'Valve 1', 'Valve 2', 'Battery_V', 'Vcc_V', 'VPD_kPa']
Targets: ['Lighting', 'Ventilation', 'Air_circulation', 'Heater', 'Fog']


## 3) Adding VPD

In [ ]:
# VPD in kPa from Temp_C and RH_pct
# (Only if these exist)
if "Temp_C" in df.columns and "RH_pct" in df.columns:
    T = df["Temp_C"].astype(float)
    RH = df["RH_pct"].astype(float)
    svp = 0.6108 * np.exp((17.27 * T) / (T + 237.3))
    df["VPD_kPa"] = svp * (1 - RH/100.0)
    if "VPD_kPa" not in X_cols:
        X_cols.append("VPD_kPa")

df[["Temp_C","RH_pct","VPD_kPa"]].head()


,Temp_C,RH_pct,VPD_kPa
0,26.7,74,0.910798
1,27.4,74,0.948966
2,28.1,73,1.026539
3,28.4,72,1.083282
4,29.0,71,1.161647


## 4) Building sequences (horizon=5)

In [ ]:
def make_sequences(df, X_cols, y_cols, seq_len=30, horizon=5):
    X = df[X_cols].astype(float).to_numpy()
    Y = df[y_cols].astype(int).to_numpy()

    # Before make_sequences(...)
    df[X_cols] = df[X_cols].astype(float)

    # time-series friendly imputation
    df[X_cols] = df[X_cols].ffill().bfill()

    # final safety: fill any remaining with median
    df[X_cols] = df[X_cols].fillna(df[X_cols].median(numeric_only=True))


    n = len(df)
    # last usable input window ends at t, label taken at t+horizon
    last_t = n - 1 - horizon
    X_seq, Y_out, ts = [], [], []

    for t in range(seq_len-1, last_t+1):
        X_seq.append(X[t-seq_len+1:t+1])
        Y_out.append(Y[t+horizon])
        ts.append(df["Timestamp"].iloc[t+horizon])

    return np.asarray(X_seq), np.asarray(Y_out), pd.to_datetime(ts)

SEQ_LEN = 30     # last 30 minutes (if 1-min sampling)
HORIZON = 5      # predict action 5 minutes ahead

X_seq, Y, y_ts = make_sequences(df, X_cols, target_cols, seq_len=SEQ_LEN, horizon=HORIZON)
print(X_seq.shape, Y.shape, y_ts.min(), y_ts.max())


(30193, 30, 16) (30193, 5) 2025-05-17 13:23:00 2025-06-08 13:12:00


## 5) Time-based split

In [ ]:
n = len(X_seq)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)

X_train, y_train = X_seq[:train_end], Y[:train_end]
X_val,   y_val   = X_seq[train_end:val_end], Y[train_end:val_end]
X_test,  y_test  = X_seq[val_end:], Y[val_end:]

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)


Train/Val/Test: (21135, 30, 16) (4529, 30, 16) (4529, 30, 16)


## 6) Scaling features

In [ ]:
scaler = StandardScaler()

# Fit on train flattened over time
X_train_flat = X_train.reshape(-1, X_train.shape[-1])
scaler.fit(X_train_flat)

def scale_3d(X):
    Xf = X.reshape(-1, X.shape[-1])
    Xs = scaler.transform(Xf)
    return Xs.reshape(X.shape)

X_train_s = scale_3d(X_train)
X_val_s   = scale_3d(X_val)
X_test_s  = scale_3d(X_test)


In [ ]:
def check_nan(name, arr):
    print(name, "shape", arr.shape,
          "nan:", np.isnan(arr).any(),
          "inf:", np.isinf(arr).any())

check_nan("X_train_s", X_train_s)
check_nan("X_val_s", X_val_s)
check_nan("X_test_s", X_test_s)


X_train_s shape (21135, 30, 16) nan: False inf: False
X_val_s shape (4529, 30, 16) nan: True inf: False
X_test_s shape (4529, 30, 16) nan: True inf: False


## 7) Per-actuator pos weights to handle imbalance

In [ ]:
def make_sample_weights(y_2d):
    # y_2d shape: (N, num_tasks)
    weights = {}
    for i, name in enumerate(target_cols):
        y = y_2d[:, i]
        pos = (y == 1).sum()
        neg = (y == 0).sum()
        # avoid divide by zero
        w_pos = (neg / max(pos, 1.0))
        w = np.where(y == 1, w_pos, 1.0).astype("float32")
        weights[name] = w
        print(f"{name}: pos={pos} neg={neg} pos_weight~{w_pos:.2f}")
    return weights

# Keras wants y as dict if outputs are dict
y_train_dict = {name: y_train[:, i].astype("float32") for i, name in enumerate(target_cols)}
y_val_dict   = {name: y_val[:, i].astype("float32")   for i, name in enumerate(target_cols)}
y_test_dict  = {name: y_test[:, i].astype("float32")  for i, name in enumerate(target_cols)}

sw_train = make_sample_weights(y_train)
sw_val   = make_sample_weights(y_val)


Lighting: pos=5693 neg=15442 pos_weight~2.71
Ventilation: pos=0 neg=21135 pos_weight~21135.00
Air_circulation: pos=0 neg=21135 pos_weight~21135.00
Heater: pos=0 neg=21135 pos_weight~21135.00
Fog: pos=9461 neg=11674 pos_weight~1.23
Lighting: pos=0 neg=4529 pos_weight~4529.00
Ventilation: pos=0 neg=4529 pos_weight~4529.00
Air_circulation: pos=0 neg=4529 pos_weight~4529.00
Heater: pos=0 neg=4529 pos_weight~4529.00
Fog: pos=2482 neg=2047 pos_weight~0.82


## 8) Multi-head GRU model

In [ ]:
tf.keras.utils.set_random_seed(42)

inp = keras.Input(shape=(SEQ_LEN, X_train_s.shape[-1]))

x = layers.GRU(64, return_sequences=True)(inp)
x = layers.GRU(32)(x)
x = layers.Dropout(0.3)(x)

outputs = {}
for name in target_cols:
    outputs[name] = layers.Dense(1, activation="sigmoid", name=name)(x)

model = keras.Model(inp, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={name: "binary_crossentropy" for name in target_cols},
    metrics={name: [keras.metrics.AUC(curve="PR", name="pr_auc")] for name in target_cols},
)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 30, 16)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_4 (GRU)         │ (None, 30, 64)    │     15,744 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_5 (GRU)         │ (None, 32)        │      9,408 │ gru_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ gru_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Air_circulation     │ (None, 1)         │         33 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Fog (Dense)         │ (None, 1)         │         33 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Heater (Dense)      │ (None, 1)         │         33 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Lighting (Dense)    │ (None, 1)         │         33 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Ventilation (Dense) │ (None, 1)         │         33 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 25,317 (98.89 KB)

 Trainable params: 25,317 (98.89 KB)

 Non-trainable params: 0 (0.00 B)

## 9) Training

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

history = model.fit(
    X_train_s, y_train_dict,
    validation_data=(X_val_s, y_val_dict, sw_val),
    sample_weight=sw_train,
    epochs=30,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 20s 138ms/step - Air_circulation_loss: 0.4732 - Air_circulation_pr_auc: 0.0000e+00 - Fog_loss: 0.3126 - Fog_pr_auc: 0.9724 - Heater_loss: 0.5161 - Heater_pr_auc: 0.0000e+00 - Lighting_loss: 0.7875 - Lighting_pr_auc: 0.5990 - Ventilation_loss: 0.4721 - Ventilation_pr_auc: 0.0000e+00 - loss: 2.5617 - val_Air_circulation_loss: nan - val_Air_circulation_pr_auc: 0.0000e+00 - val_Fog_loss: nan - val_Fog_pr_auc: 0.9640 - val_Heater_loss: nan - val_Heater_pr_auc: 0.0000e+00 - val_Lighting_loss: nan - val_Lighting_pr_auc: 0.0000e+00 - val_Ventilation_loss: nan - val_Ventilation_pr_auc: 0.0000e+00 - val_loss: nan - learning_rate: 0.0010
Epoch 2/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 10s 113ms/step - Air_circulation_loss: 0.0568 - Air_circulation_pr_auc: 0.0000e+00 - Fog_loss: 0.1521 - Fog_pr_auc: 0.9906 - Heater_loss: 0.0625 - Heater_pr_auc: 0.0000e+00 - Lighting_loss: 0.2398 - Lighting_pr_auc: 0.9626 - Ventilation_loss: 0.0542 - Ventilation_pr_auc: 0.0000e+00 - loss

## 10) Evaluation

In [ ]:
def eval_task(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        # "pr_auc": average_precision_score(y_true, y_prob),
        "cm": confusion_matrix(y_true, y_pred, labels=[0, 1]),
    }

pred = model.predict(X_test_s, verbose=0)

results = {}
for i, name in enumerate(target_cols):
    y_true = y_test[:, i]
    y_prob = pred[name].reshape(-1)
    results[name] = eval_task(y_true, y_prob, thr=0.5)

for name, r in results.items():
    print("\n===", name, "===")
    print({k: v for k, v in r.items() if k != "cm"})
    print("Confusion matrix:\n", r["cm"])



=== Lighting ===
{'balanced_acc': np.float64(0.5961580922941047), 'f1': 0.0, 'recall': 0.0}
Confusion matrix:
 [[2700 1829]
 [   0    0]]

=== Ventilation ===
{'balanced_acc': np.float64(1.0), 'f1': 0.0, 'recall': 0.0}
Confusion matrix:
 [[4529    0]
 [   0    0]]

=== Air_circulation ===
{'balanced_acc': np.float64(1.0), 'f1': 0.0, 'recall': 0.0}
Confusion matrix:
 [[4529    0]
 [   0    0]]

=== Heater ===
{'balanced_acc': np.float64(1.0), 'f1': 0.0, 'recall': 0.0}
Confusion matrix:
 [[4529    0]
 [   0    0]]

=== Fog ===
{'balanced_acc': np.float64(0.9419818164676856), 'f1': 0.9527962085308057, 'recall': 0.9617298124760811}
Confusion matrix:
 [[1767  149]
 [ 100 2513]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
